# House Stage 1 regression: finalists, rolling 2018–2022

The finalist specifications from the ablation notebooks, in the same expanding-window backtest. Each window refits every model on prior elections only and scores the next one:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

**Base model:** `BASE_FORMULA`, the classic formula plus `partisan_lean:inc_signed`, by OLS with `inc_signed` **fixed at `FIXED_INC` (2.5)**. That is `run_ols_fixed(..., fixed={"inc_signed": FIXED_INC})`: OLS of `adj_margin_2p_norm − FIXED_INC · inc_signed` on the other terms, the interaction included, with `FIXED_INC · inc_signed` added back to the predictions.

**Finalists**, each the base plus:

- `fundraising_advantage`
- `hispanic`
- `college`
- `hispanic` + `college`
- `fundraising_advantage` + the best of the three demographic models
- all three additional predictors (`ALL_PREDICTORS`: `fundraising_advantage`, `hispanic`, `college`)
- generic-ballot momentum, `gb_momentum_30d_raw`: the raw generic-ballot margin on election eve minus the margin 30 days earlier (`data-processing/house/01-generic-ballot.ipynb`), from polls published before the election. It is one national value per year, so its coefficient is estimated from the training years alone (6–8). With `gen_margin_raw` in the model, a positive coefficient means late movement keeps going past the last poll; a negative one means it partly reverses.
- the generic ballot's 30-day EWMA, `gb_margin_30d_ewma_raw` (an exponentially weighted mean of the daily margin, span 30 days, through election eve), alongside `gen_margin_raw` (`plus_gb_ewma`)

and two with the generic ballot **replaced**, not added: `gen_margin_raw` swapped for its 30-day EWMA (`gb_ewma_instead`), or for the two-party EWMA, `gb_margin_30d_ewma_2p_norm` (`gb_ewma_2p_instead`). These test a smoothed generic ballot against the election-eve reading.

and the same set **without the interaction**, named `no_lean_x_inc_*`: the classic formula (`FORMULA`) plus each subset, with `inc_signed` still fixed at `FIXED_INC` but no `partisan_lean:inc_signed`. `no_lean_x_inc` alone is the classic formula with incumbency fixed. Every Δ is against the base, which has the interaction, so comparing a model with its `no_lean_x_inc_` twin shows what the interaction adds.

Every model is fit with the same fixed incumbency coefficient and on the same training and evaluation rows (`FINALIST_SHARED_COLUMNS`). The evaluation subsets are the other notebooks': valid, `competitive_ex_ante` (|lean + generic ballot| ≤ 10) and `competitive_ex_post` (|margin| ≤ 7.5). 2024 is never used (`USE_HOLDOUT = False`).

**Caveats:**

- **Choosing the best demographic model:** it is picked from these same three eval years (lowest mean MAE on `BEST_DEMOGRAPHIC_SUBSET`, ties by RMSE), so the last finalist's scores are somewhat optimistic.
- **FEC timing:** totals run through December 31, after the election.

In [1]:
%load_ext autoreload
%autoreload 2
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import evaluation, tracking
from experiments.house.run import run_ols_fixed

In [2]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

In [3]:
FORMULA = "adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed"
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
REQUIRE_COMPLETE = []   # extra columns a row must have (FINALIST_SHARED_COLUMNS below adds every model's columns)
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

In [4]:
FIXED_TERM = "inc_signed"
FIXED_INC = 2.5  # imposed incumbency coefficient in every model, points of D margin
BASE_FORMULA = FORMULA + " + partisan_lean:inc_signed"
BASE_NAME = f"base_inc_fixed_{FIXED_INC:g}_lean_x_inc"

FUNDRAISING = ["fundraising_advantage"]
GB_MOMENTUM = ["gb_momentum_30d_raw"]  # election-eve generic-ballot margin minus 30 days earlier, raw
GB_EWMA = ["gb_margin_30d_ewma_raw"]   # 30-day EWMA of the raw generic-ballot margin, through election eve
GB_SWAPS = {  # model: the generic-ballot column that replaces gen_margin_raw
    "gb_ewma_instead": "gb_margin_30d_ewma_raw",
    "gb_ewma_2p_instead": "gb_margin_30d_ewma_2p_norm",
}
DEMOGRAPHIC_MODELS = {  # model: the demographic columns it adds to the base
    "plus_hispanic": ["hispanic"],
    "plus_college": ["college"],
    "plus_hispanic_college": ["hispanic", "college"],
}
ALL_PREDICTORS = [*FUNDRAISING, "hispanic", "college"]
NO_INTERACTION = "no_lean_x_inc"  # prefix of the models without partisan_lean:inc_signed (FORMULA, not BASE_FORMULA)
BEST_DEMOGRAPHIC_SUBSET = "valid"  # the subset the best demographic model is chosen on (by RANK_BY, ties TIEBREAK)


def with_terms(columns, base=BASE_FORMULA):
    """`base` (BASE_FORMULA, with the interaction, or FORMULA, without) plus `columns`."""
    return base + "".join(f" + {column}" for column in columns)


SUBSETS = {  # model suffix: the additional predictors it adds (the fundraising + best demographic is added later)
    "plus_fundraising": FUNDRAISING,
    **DEMOGRAPHIC_MODELS,
    "plus_all_predictors": ALL_PREDICTORS,
    "plus_gb_momentum": GB_MOMENTUM,
    "plus_gb_ewma": GB_EWMA,
}
FINALISTS = {  # model: formula
    BASE_NAME: BASE_FORMULA,  # the base
    **{name: with_terms(columns) for name, columns in SUBSETS.items()},
    NO_INTERACTION: FORMULA,  # the classic formula with incumbency fixed
    **{f"{NO_INTERACTION}_{name}": with_terms(columns, base=FORMULA) for name, columns in SUBSETS.items()},
    # the generic ballot replaced by a smoothed version, with and without the interaction
    **{name: BASE_FORMULA.replace("gen_margin_raw", column) for name, column in GB_SWAPS.items()},
    **{f"{NO_INTERACTION}_{name}": FORMULA.replace("gen_margin_raw", column) for name, column in GB_SWAPS.items()},
}
# every column any finalist uses: all models share the same rows
FINALIST_SHARED_COLUMNS = evaluation.shared_columns(formulas=FINALISTS.values(), columns=predictors.columns)
print("shared columns:", FINALIST_SHARED_COLUMNS)
for name, formula in FINALISTS.items():
    print(f"{name:40s} {formula}")

shared columns: ['gen_margin_raw', 'partisan_lean', 'inc_signed', 'fundraising_advantage', 'hispanic', 'college', 'gb_momentum_30d_raw', 'gb_margin_30d_ewma_raw', 'gb_margin_30d_ewma_2p_norm']
base_inc_fixed_2.5_lean_x_inc            adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed + partisan_lean:inc_signed
plus_fundraising                         adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed + partisan_lean:inc_signed + fundraising_advantage
plus_hispanic                            adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed + partisan_lean:inc_signed + hispanic
plus_college                             adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed + partisan_lean:inc_signed + college
plus_hispanic_college                    adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed + partisan_lean:inc_signed + hispanic + college
plus_all_predictors                      adj_margin_2p_norm ~ gen_margin_raw

In [5]:
# Every window: the base, fundraising, and the three demographic models, all on the same rows. The missing-predictor
# warnings (FEC has no data for a few winners) repeat for every window and model; each run's exclusions are in its
# .excluded and .counts
with warnings.catch_warnings(action="ignore"):
    rolling = {
        eval_year: {
            "finalists": [
                run_ols_fixed(
                    formula=formula,
                    train_years=train_years,
                    eval_years=[eval_year],
                    predict_years=[],
                    fixed={FIXED_TERM: FIXED_INC},
                    predictors=predictors,
                    targets=targets,
                    use_holdout=USE_HOLDOUT,
                    require_complete=FINALIST_SHARED_COLUMNS,
                    race_types=EVAL_RACE_TYPES,
                    competitive=COMPETITIVE,
                    name=name,
                    save=False,
                )
                for name, formula in FINALISTS.items()
            ]
        }
        for eval_year, train_years in ROLLING_WINDOWS.items()
    }

In [6]:
# The best demographic model: lowest mean (over the windows) RANK_BY on BEST_DEMOGRAPHIC_SUBSET, ties by TIEBREAK
def mean_score(name, metric):
    """`metric` of model `name` on BEST_DEMOGRAPHIC_SUBSET, averaged over the windows."""
    runs = [next(run for run in blocks["finalists"] if run.config["name"] == name) for blocks in rolling.values()]
    return sum(run.metrics.loc[(BEST_DEMOGRAPHIC_SUBSET, "all"), metric] for run in runs) / len(runs)


demographic_scores = pd.DataFrame(
    {name: {metric: mean_score(name, metric) for metric in [RANK_BY, TIEBREAK]} for name in DEMOGRAPHIC_MODELS}
).T.sort_values([RANK_BY, TIEBREAK])
BEST_DEMOGRAPHIC = demographic_scores.index[0]
COMBINED_NAME = f"plus_fundraising_{BEST_DEMOGRAPHIC.removeprefix('plus_')}"
COMBINED = {  # with and without the interaction; the same best demographic model for both
    COMBINED_NAME: with_terms([*FUNDRAISING, *DEMOGRAPHIC_MODELS[BEST_DEMOGRAPHIC]]),
    f"{NO_INTERACTION}_{COMBINED_NAME}": with_terms([*FUNDRAISING, *DEMOGRAPHIC_MODELS[BEST_DEMOGRAPHIC]], base=FORMULA),
}
FINALISTS |= COMBINED
print(f"best demographic model: {BEST_DEMOGRAPHIC}")
for name, formula in COMBINED.items():
    print(f"{name:40s} {formula}")
demographic_scores.round(3)

best demographic model: plus_hispanic
plus_fundraising_hispanic                adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed + partisan_lean:inc_signed + fundraising_advantage + hispanic
no_lean_x_inc_plus_fundraising_hispanic  adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed + fundraising_advantage + hispanic


,mae,rmse
plus_hispanic,5.893,7.642
plus_college,5.909,7.550
plus_hispanic_college,6.005,7.733


In [7]:
# fundraising + the best demographic model (with and without the interaction), in every window, on the same rows
with warnings.catch_warnings(action="ignore"):
    for eval_year, train_years in ROLLING_WINDOWS.items():
        rolling[eval_year]["finalists"] += [
            run_ols_fixed(
                formula=formula,
                train_years=train_years,
                eval_years=[eval_year],
                predict_years=[],
                fixed={FIXED_TERM: FIXED_INC},
                predictors=predictors,
                targets=targets,
                use_holdout=USE_HOLDOUT,
                require_complete=FINALIST_SHARED_COLUMNS,
                race_types=EVAL_RACE_TYPES,
                competitive=COMPETITIVE,
                name=name,
                save=False,
            )
            for name, formula in COMBINED.items()
        ]

## Results

For each section (ALL, EX_ANTE and EX_POST), there are two tables. The first is **by eval year**, each window on its own. The second is the **mean over 2018, 2020 and 2022**, each window weighted equally. Models are ranked by mean MAE, with ties broken by RMSE.

- **Metrics:** `n_train`, `n`, MAE, RMSE, bias (mean residual, actual − predicted: + means Democrats outperformed the forecast), accuracy, precision D/R and misses.
- **Mean table extras:** `|bias|` (the mean of each window's |bias|) and `mae_<year>` for each window.
- **Δ columns:** model − the base in the same window, on the same rows. ✓ (green) marks an improvement and ✗ (red) a regression.

In [8]:
for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)[0]
         for year, blocks in rolling.items()}, names=["eval_year"]).droplevel("block")
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}, "
                     f"ranked by {RANK_BY} (ties: {TIEBREAK})"))
    display(tracking.style_best_of(mean.droplevel("block")))

### ALL (`valid`): by eval year

### ALL (`valid`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
base_inc_fixed_2.5_lean_x_inc,2680,400,5.75,7.39,-0.01,2.11,95.57,95.78,95.61,18,4.43,6.29,5.75,5.19,-,-,-,-,-,-,-,-
no_lean_x_inc_plus_gb_momentum,2680,400,5.52,7.16,-0.74,1.28,94.91,96.15,93.86,20,5.09,5.93,5.75,4.89,-0.22 ✓,-0.22 ✓,-0.66 ✗,+0.37 ✓,-1.75 ✗,+3 ✗,+0.66 ✗,-0.83 ✓
plus_gb_momentum,2680,400,5.59,7.21,-0.25,1.49,95.66,95.77,95.75,17,4.34,6.10,5.54,5.14,-0.15 ✓,-0.17 ✓,+0.08 ✓,-0.01 ✗,+0.13 ✓,-0 ✓,-0.08 ✓,-0.62 ✓
no_lean_x_inc_gb_ewma_instead,2680,400,5.67,7.30,-0.42,1.91,94.65,96.14,93.44,21,5.35,6.16,5.93,4.93,-0.07 ✓,-0.09 ✓,-0.92 ✗,+0.36 ✓,-2.17 ✗,+4 ✗,+0.92 ✗,-0.20 ✓
no_lean_x_inc_gb_ewma_2p_instead,2680,400,5.68,7.29,-0.24,1.94,94.48,96.13,93.16,22,5.52,6.26,5.85,4.93,-0.07 ✓,-0.10 ✓,-1.09 ✗,+0.35 ✓,-2.45 ✗,+4 ✗,+1.09 ✗,-0.17 ✓
no_lean_x_inc,2680,400,5.69,7.31,-0.48,1.91,94.65,96.14,93.44,21,5.35,6.16,5.97,4.93,-0.06 ✓,-0.07 ✓,-0.92 ✗,+0.36 ✓,-2.17 ✗,+4 ✗,+0.92 ✗,-0.20 ✓
no_lean_x_inc_plus_gb_ewma,2680,400,5.69,7.32,-0.50,1.89,94.65,96.14,93.44,21,5.35,6.18,5.98,4.92,-0.05 ✓,-0.07 ✓,-0.92 ✗,+0.36 ✓,-2.17 ✗,+4 ✗,+0.92 ✗,-0.22 ✓
gb_ewma_instead,2680,400,5.73,7.37,+0.05,2.11,95.57,95.78,95.61,18,4.43,6.30,5.71,5.19,-0.01 ✓,-0.02 ✓,+0.00,+0.00,+0.00,+0,+0.00,+0.00 ✗
gb_ewma_2p_instead,2680,400,5.73,7.37,+0.22,2.13,95.41,95.77,95.30,18,4.59,6.39,5.62,5.18,-0.01 ✓,-0.02 ✓,-0.17 ✗,-0.01 ✗,-0.32 ✗,+1 ✗,+0.17 ✗,+0.02 ✗


### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): by eval year

### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
base_inc_fixed_2.5_lean_x_inc,2680,91,5.12,6.46,-0.13,2.09,84.44,85.49,84.33,14,15.56,5.11,4.99,5.25,-,-,-,-,-,-,-,-
no_lean_x_inc_plus_fundraising_hispanic,2680,91,4.64,5.73,-1.54,1.54,84.74,80.83,87.39,14,15.26,3.97,4.93,5.01,-0.48 ✓,-0.73 ✓,+0.30 ✓,-4.66 ✗,+3.06 ✓,-0 ✓,-0.30 ✓,-0.55 ✓
no_lean_x_inc_plus_fundraising,2680,91,4.64,5.72,-1.54,1.54,84.71,80.96,87.45,14,15.29,3.94,4.96,5.02,-0.48 ✓,-0.74 ✓,+0.28 ✓,-4.53 ✗,+3.11 ✓,-0 ✓,-0.28 ✓,-0.55 ✓
no_lean_x_inc_plus_all_predictors,2680,91,4.64,5.71,-1.30,1.39,83.64,80.51,85.61,15,16.36,4.09,4.88,4.95,-0.48 ✓,-0.75 ✓,-0.80 ✗,-4.98 ✗,+1.28 ✓,+1 ✗,+0.80 ✗,-0.71 ✓
plus_fundraising,2680,91,4.65,5.74,-1.59,1.59,84.71,80.96,87.45,14,15.29,3.93,5.00,5.03,-0.46 ✓,-0.72 ✓,+0.28 ✓,-4.53 ✗,+3.11 ✓,-0 ✓,-0.28 ✓,-0.50 ✓
plus_fundraising_hispanic,2680,91,4.65,5.75,-1.60,1.60,84.38,80.21,87.38,14,15.62,3.97,4.97,5.03,-0.46 ✓,-0.71 ✓,-0.06 ✗,-5.28 ✗,+3.05 ✓,+0,+0.06 ✗,-0.50 ✓
plus_all_predictors,2680,91,4.66,5.73,-1.36,1.47,83.71,80.45,86.14,15,16.29,4.09,4.92,4.97,-0.46 ✓,-0.73 ✓,-0.72 ✗,-5.03 ✗,+1.81 ✓,+1 ✗,+0.72 ✗,-0.62 ✓
plus_gb_momentum,2680,91,4.96,6.30,-0.34,1.77,84.77,85.77,84.72,14,15.23,4.77,4.81,5.31,-0.15 ✓,-0.15 ✓,+0.33 ✓,+0.28 ✓,+0.39 ✓,-0 ✓,-0.33 ✓,-0.33 ✓
no_lean_x_inc_plus_gb_momentum,2680,91,5.09,6.37,+0.91,1.22,81.42,86.07,77.85,17,18.58,5.13,4.85,5.30,-0.03 ✓,-0.08 ✓,-3.02 ✗,+0.58 ✓,-6.48 ✗,+3 ✗,+3.02 ✗,-0.87 ✓


### EX_POST_COMPETITIVE (`competitive_ex_post`): by eval year

### EX_POST_COMPETITIVE (`competitive_ex_post`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
base_inc_fixed_2.5_lean_x_inc,2680,60,6.44,8.39,+0.42,2.72,72.74,74.72,75.25,16,27.26,7.91,5.88,5.52,-,-,-,-,-,-,-,-
no_lean_x_inc_plus_fundraising,2680,60,5.02,6.45,-1.47,2.07,72.42,70.73,77.42,16,27.58,4.41,5.47,5.18,-1.42 ✓,-1.94 ✓,-0.32 ✗,-3.99 ✗,+2.16 ✓,+0,+0.32 ✗,-0.65 ✓
plus_fundraising,2680,60,5.03,6.45,-1.53,2.16,72.42,70.73,77.42,16,27.58,4.38,5.52,5.19,-1.41 ✓,-1.94 ✓,-0.32 ✗,-3.99 ✗,+2.16 ✓,+0,+0.32 ✗,-0.56 ✓
no_lean_x_inc_plus_fundraising_hispanic,2680,60,5.08,6.56,-1.53,2.06,72.97,71.27,77.50,16,27.03,4.54,5.52,5.17,-1.36 ✓,-1.83 ✓,+0.23 ✓,-3.45 ✗,+2.25 ✓,-0 ✓,-0.23 ✓,-0.66 ✓
no_lean_x_inc_plus_all_predictors,2680,60,5.08,6.54,-1.28,2.00,71.78,71.11,76.08,17,28.22,4.71,5.44,5.08,-1.36 ✓,-1.85 ✓,-0.96 ✗,-3.60 ✗,+0.83 ✓,+0 ✗,+0.96 ✗,-0.71 ✓
plus_fundraising_hispanic,2680,60,5.10,6.57,-1.59,2.15,72.42,70.73,77.42,16,27.58,4.51,5.58,5.19,-1.34 ✓,-1.82 ✓,-0.32 ✗,-3.99 ✗,+2.16 ✓,+0,+0.32 ✗,-0.57 ✓
plus_all_predictors,2680,60,5.10,6.55,-1.35,2.10,71.93,71.19,76.46,17,28.07,4.69,5.51,5.09,-1.34 ✓,-1.84 ✓,-0.81 ✗,-3.52 ✗,+1.20 ✓,+0 ✗,+0.81 ✗,-0.62 ✓
no_lean_x_inc_plus_gb_momentum,2680,60,6.22,8.05,+1.70,2.08,69.97,74.50,68.23,18,30.03,7.45,5.63,5.57,-0.22 ✓,-0.34 ✓,-2.77 ✗,-0.22 ✗,-7.02 ✗,+2 ✗,+2.77 ✗,-0.64 ✓
plus_gb_momentum,2680,60,6.26,8.21,+0.22,2.39,73.23,75.02,74.89,16,26.77,7.47,5.74,5.58,-0.17 ✓,-0.18 ✓,+0.49 ✓,+0.30 ✓,-0.36 ✗,-0 ✓,-0.49 ✓,-0.33 ✓


## Winner calls: accuracy and precision

Each model's winner calls side by side for ALL, EX_ANTE and EX_POST. Winners come from the sign of each margin, and races where either margin is exactly 0 are left out.

- **`accuracy_pct`:** the share of races whose winner was called correctly.
- **`precision_D_pct`:** of the races predicted D, the share the Democrat won.
- **`precision_R_pct`:** of the races predicted R, the share the Republican won.
- **Δ columns:** model − the base, on the same races. ✓ marks an improvement and ✗ a regression.

The first table is the mean over the three eval years, each weighted equally. The second is by eval year. The competitive subsets are small (`n` races per window, about 91 ex-ante and 60 ex-post), so one race moves a window's accuracy by about 1.1 points ex-ante and 1.7 ex-post.

In [9]:
CALL_METRICS = ["n", "accuracy_pct", "precision_D_pct", "precision_R_pct"]
CALL_COLUMNS = [*CALL_METRICS, *(f"Δ{metric}" for metric in CALL_METRICS[1:])]


def winner_calls(table: pd.DataFrame) -> pd.DataFrame:
    """A best_of_blocks / rolling_best_of table's winner-call columns."""
    return table.reindex(columns=CALL_COLUMNS)


mean_calls = pd.concat(
    {section: winner_calls(tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=None,
                                                    tiebreak=TIEBREAK)[0].droplevel("block"))
     for section, subset in SUMMARY_SECTIONS.items()}, axis=1).reindex(list(FINALISTS))
display(Markdown(f"### Winner calls: mean of {', '.join(map(str, ROLLING_WINDOWS))}"))
display(tracking.style_best_of(mean_calls))

calls_by_year = pd.concat(
    {section: pd.concat({year: winner_calls(tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=None,
                                                                    tiebreak=TIEBREAK)[0].droplevel("block"))
                         .reindex(list(FINALISTS)) for year, blocks in rolling.items()}, names=["eval_year"])
     for section, subset in SUMMARY_SECTIONS.items()}, axis=1)
display(Markdown("### Winner calls: by eval year"))
display(tracking.style_best_of(calls_by_year))

### Winner calls: mean of 2018, 2020, 2022

### Winner calls: by eval year

## Coefficients

Each model's coefficients by window, then averaged over the three. `inc_signed` is the imposed `FIXED_INC` in every model (it has no standard error). Everything else is estimated around it, `partisan_lean:inc_signed` included.

In [10]:
# coefficients by window (rows: eval year and model, columns: terms)
pd.concat({eval_year: pd.DataFrame({run.config["name"]: run.coefficients["coef"] for run in blocks["finalists"]}).T
           for eval_year, blocks in rolling.items()}, names=["eval_year", "model"]).round(3)

term                                               Intercept  college  \
eval_year model                                                         
2018      base_inc_fixed_2.5_lean_x_inc                1.866      NaN   
          plus_fundraising                             0.371      NaN   
          plus_hispanic                                0.764      NaN   
          plus_college                                 5.495   -0.124   
          plus_hispanic_college                        4.009   -0.105   
...                                                      ...      ...   
2022      gb_ewma_2p_instead                           0.846      NaN   
          no_lean_x_inc_gb_ewma_instead               -1.052      NaN   
          no_lean_x_inc_gb_ewma_2p_instead            -1.043      NaN   
          plus_fundraising_hispanic                    0.146      NaN   
          no_lean_x_inc_plus_fundraising_hispanic     -0.007      NaN   

term                                               fundraising_advantage  \
eval_year model                                                            
2018      base_inc_fixed_2.5_lean_x_inc                              NaN   
          plus_fundraising                                         4.871   
          plus_hispanic                                              NaN   
          plus_college                                               NaN   
          plus_hispanic_college                                      NaN   
...                                                                  ...   
2022      gb_ewma_2p_instead                                         NaN   
          no_lean_x_inc_gb_ewma_instead                              NaN   
          no_lean_x_inc_gb_ewma_2p_instead                           NaN   
          plus_fundraising_hispanic                                4.347   
          no_lean_x_inc_plus_fundraising_hispanic                  4.366   

term                                               gb_margin_30d_ewma_2p_norm  \
eval_year model                                                                 
2018      base_inc_fixed_2.5_lean_x_inc                                   NaN   
          plus_fundraising                                                NaN   
          plus_hispanic                                                   NaN   
          plus_college                                                    NaN   
          plus_hispanic_college                                           NaN   
...                                                                       ...   
2022      gb_ewma_2p_instead                                            0.776   
          no_lean_x_inc_gb_ewma_instead                                   NaN   
          no_lean_x_inc_gb_ewma_2p_instead                              0.766   
          plus_fundraising_hispanic                                       NaN   
          no_lean_x_inc_plus_fundraising_hispanic                         NaN   

term                                               gb_margin_30d_ewma_raw  \
eval_year model                                                             
2018      base_inc_fixed_2.5_lean_x_inc                               NaN   
          plus_fundraising                                            NaN   
          plus_hispanic                                               NaN   
          plus_college                                                NaN   
          plus_hispanic_college                                       NaN   
...                                                                   ...   
2022      gb_ewma_2p_instead                                          NaN   
          no_lean_x_inc_gb_ewma_instead                             0.862   
          no_lean_x_inc_gb_ewma_2p_instead                            NaN   
          plus_fundraising_hispanic                                   NaN   
          no_lean_x_inc_plus_fundraising_hispanic                     NaN   

term 

In [11]:
# each model's coefficients averaged over the windows
_, mean_coefficients = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=None, tiebreak=TIEBREAK)
mean_coefficients.droplevel("block").round(3)

,Intercept,gen_margin_raw,partisan_lean,partisan_lean:inc_signed,inc_signed,fundraising_advantage,hispanic,college,gb_momentum_30d_raw,gb_margin_30d_ewma_raw,gb_margin_30d_ewma_2p_norm
run,,,,,,,,,,,
base_inc_fixed_2.5_lean_x_inc,1.732,0.798,1.068,-0.106,2.5,NaN,NaN,NaN,NaN,NaN,NaN
no_lean_x_inc_plus_gb_momentum,-2.412,1.279,1.044,NaN,2.5,NaN,NaN,NaN,-1.550,NaN,NaN
plus_gb_momentum,-0.389,1.258,1.067,-0.104,2.5,NaN,NaN,NaN,-1.462,NaN,NaN
no_lean_x_inc_gb_ewma_instead,-0.916,NaN,1.044,NaN,2.5,NaN,NaN,NaN,NaN,0.882,NaN
no_lean_x_inc_gb_ewma_2p_instead,-0.921,NaN,1.044,NaN,2.5,NaN,NaN,NaN,NaN,NaN,0.779
no_lean_x_inc,-0.215,0.791,1.044,NaN,2.5,NaN,NaN,NaN,NaN,NaN,NaN
no_lean_x_inc_plus_gb_ewma,0.360,1.351,1.044,NaN,2.5,NaN,NaN,NaN,NaN,-0.644,NaN
gb_ewma_instead,1.009,NaN,1.067,-0.105,2.5,NaN,NaN,NaN,NaN,0.889,NaN
gb_ewma_2p_instead,0.998,NaN,1.067,-0.105,2.5,NaN,NaN,NaN,NaN,NaN,0.785
